# Global Political Trust Dashboard

Scope & flow: overall descriptive stats for all institutional trust items (ti1–ti12), full-sample distributions, variation across countries and years, demographic splits (gender/education/income), plus focused visuals for the selected country set. The dashboard ends with ordered descriptive tables and explanatory notes.

This notebook loads `GBS2.9-SAMPLE.sav` (political trust survey), cleans trust variables, produces descriptive stats, cross-country/year comparisons, demographic slices, and builds a Plotly HTML dashboard saved to `report/gbs_pt_html.html`. Scale notes: values 1–4 indicate increasing trust (1=None at all, 2=Not very much, 3=Quite a lot, 4=A great deal). Missing/invalid codes (-1/0/7/8/9) and invalid labels (missing/NA/decline/don’t know/do not understand the question/etc.) are excluded from averages and splits.


In [280]:
# Environment & paths
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import pyreadstat


# Locate data directory (robust to different working dirs)
def locate_data_dir() -> Path:
    candidates = [
        Path.cwd() / "Lab Survey AI Analysis/data",
        Path.cwd() / "data",
        Path.cwd().parent / "data",
        Path.cwd().parent / "Lab Survey AI Analysis/data",
    ]
    for c in candidates:
        candidate = c / "GBS2.9-SAMPLE.sav"
        if candidate.exists():
            return c
    raise FileNotFoundError("GBS2.9-SAMPLE.sav not found; please adjust DATA_DIR or working directory.")


DATA_DIR = locate_data_dir()
PROJECT_ROOT = DATA_DIR.parent
SPSS_PATH = DATA_DIR / "GBS2.9-SAMPLE.sav"
CODEBOOK_PATH = DATA_DIR / "GBS2.9_codebook.xlsx"
OUTPUT_HTML = PROJECT_ROOT / "report/gbs_pt_html.html"
OUTPUT_HTML.parent.mkdir(parents=True, exist_ok=True)

# Plotly palette & template (pastel, minimal)
PALETTE = px.colors.qualitative.Pastel
px.defaults.template = "simple_white"
px.defaults.color_discrete_sequence = PALETTE

print(f"Working directory: {Path.cwd()}")
print(f"Data file: {SPSS_PATH.resolve()} (exists: {SPSS_PATH.exists()})")
print(f"Output path: {OUTPUT_HTML.resolve()}")


Working directory: /Users/lisic/Library/CloudStorage/GoogleDrive-u3629075@connect.hku.hk/我的云端硬盘/Lab Survey AI Analysis
Data file: /Users/lisic/Library/CloudStorage/GoogleDrive-u3629075@connect.hku.hk/我的云端硬盘/Lab Survey AI Analysis/data/GBS2.9-SAMPLE.sav (exists: True)
Output path: /Users/lisic/Library/CloudStorage/GoogleDrive-u3629075@connect.hku.hk/我的云端硬盘/Lab Survey AI Analysis/report/gbs_pt_html.html


In [281]:
# Load SPSS data & clean
import textwrap

survey_df, survey_meta = pyreadstat.read_sav(SPSS_PATH, apply_value_formats=False)
var_labels = survey_meta.column_names_to_labels
value_labels = survey_meta.variable_value_labels

# Political trust variables: institutional ti1–ti12 only
TRUST_VARS = [f"ti{i}" for i in range(1, 13) if f"ti{i}" in survey_df.columns]
trust_map = {c: var_labels.get(c, c) for c in TRUST_VARS}
trust_map_with_id = {c: f"{c} – {trust_map.get(c, c)}" for c in TRUST_VARS}

# Key value label maps
country_map = value_labels.get("country", {})
region_map = value_labels.get("region", {})
gender_map = value_labels.get("se1", {})
educ_map = value_labels.get("se3", {})
income_map = value_labels.get("se6", {})

# Missing/invalid codes to drop from averages
MISSING_CODES = {-1, 0, 7, 8, 9}
INVALID_LABEL_RE = r"(missing|not applicable|n/?a|no answer|no response|refus|declin|don.?t know|do\s*not\s*under\s*stand.*question|can.?t choose|cant choose|unknown|other|不懂|不明|不適用|不适用|不知道)"


def clean_trust(series: pd.Series) -> pd.Series:
    """Replace special codes with NaN so stats exclude invalid values."""
    return series.replace(list(MISSING_CODES), np.nan)


def wrap_label(text: str, width: int = 18) -> str:
    return "<br>".join(textwrap.wrap(text, width=width)) if isinstance(text, str) else text


def clean_category(series: pd.Series) -> pd.Series:
    mask = series.astype(str).str.strip().str.lower().str.contains(INVALID_LABEL_RE, na=False)
    return series.where(~mask)

# Preserve ti1–ti12 order for wrapped labels
VAR_ORDER_WRAPPED = [wrap_label(trust_map_with_id[v]) for v in TRUST_VARS]

# Rename and clean
survey_df = survey_df.rename(columns={"se2": "age"})
survey_df[TRUST_VARS] = survey_df[TRUST_VARS].apply(clean_trust)
survey_df["age"] = survey_df["age"].mask(lambda s: s <= 0)

survey_df = survey_df.assign(
    country_name=survey_df["country"].map(country_map),
    region_name=survey_df["region"].map(region_map),
    gender=survey_df["se1"].map(gender_map),
    education=survey_df["se3"].map(educ_map),
    income=survey_df["se6"].map(income_map),
)

# Drop invalid labels in socio-economic fields
survey_df["gender"] = clean_category(survey_df["gender"])
survey_df["education"] = clean_category(survey_df["education"])
survey_df["income"] = clean_category(survey_df["income"])

id_cols = ["region_name", "country_name", "country", "year", "gender", "education", "income", "age"]
trust_long = survey_df.melt(id_vars=id_cols, value_vars=TRUST_VARS, var_name="variable", value_name="score")
trust_long["variable_label"] = trust_long["variable"].map(trust_map)
trust_long["variable_label_with_id"] = trust_long["variable"].map(trust_map_with_id)
trust_long["variable_label_wrapped"] = trust_long["variable_label_with_id"].map(wrap_label)
trust_long = trust_long.dropna(subset=["score", "country_name", "year"])
trust_long["year_int"] = trust_long["year"].astype(int)

# Country selections (target 8 with regions)
HEAT_COUNTRIES = [
    "Mainland China",  # 亚洲晴雨表 (ABS3)
    "India",           # 南亚晴雨表 (SAB)
    "Brazil",          # 拉丁晴雨表 (LATINO)
    "Nigeria",         # 非洲晴雨表 (AFRO5)
    "Russia",          # 欧亚晴雨表 (EURASIA)
    "Egypt",           # 阿拉伯晴雨表 (ARAB3)
    "South Korea",     # 亚洲晴雨表 (ABS3)
    "South Africa",    # 非洲晴雨表 (AFRO5)
]
HEAT_COUNTRIES = [c for c in HEAT_COUNTRIES if c in trust_long["country_name"].unique()]

# Representative gender/age variables (institutional dimensions)
VAR_GENDER_AGE = {
    "ti1": "Central government",
    "ti3": "Parliament",
    "ti5": "Courts",
    "ti9": "Police",
    "ti8": "Military",
    "ti7": "Political parties",
}
VAR_GENDER_AGE_KEYS = [v for v in VAR_GENDER_AGE.keys() if v in TRUST_VARS]

# Age countries mirror heatmap selection
AGE_COUNTRIES = HEAT_COUNTRIES

print(f"Rows: {len(survey_df):,}; trust vars: {len(TRUST_VARS)} → {TRUST_VARS}")
print(f"Heatmap countries: {HEAT_COUNTRIES}")
print(f"Age slice countries: {AGE_COUNTRIES}")


/var/folders/57/jr420k75115292ssnr17qph00000gn/T/ipykernel_2581/981129432.py:35: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = series.astype(str).str.strip().str.lower().str.contains(INVALID_LABEL_RE, na=False)
/var/folders/57/jr420k75115292ssnr17qph00000gn/T/ipykernel_2581/981129432.py:35: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = series.astype(str).str.strip().str.lower().str.contains(INVALID_LABEL_RE, na=False)
/var/folders/57/jr420k75115292ssnr17qph00000gn/T/ipykernel_2581/981129432.py:35: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = series.astype(str).str.strip().str.lower().str.contains(INVALID_LABEL_RE, na=False)


Rows: 13,819; trust vars: 12 → ['ti1', 'ti2', 'ti3', 'ti4', 'ti5', 'ti6', 'ti7', 'ti8', 'ti9', 'ti10', 'ti11', 'ti12']
Heatmap countries: ['Mainland China', 'India', 'Brazil', 'Nigeria', 'Russia', 'Egypt', 'South Africa']
Age slice countries: ['Mainland China', 'India', 'Brazil', 'Nigeria', 'Russia', 'Egypt', 'South Africa']


In [282]:
# Descriptive statistics & aggregates
missing_counts = survey_df[TRUST_VARS].isna().sum().rename("missing")
valid_counts = survey_df[TRUST_VARS].notna().sum().rename("n_valid")

score_counts = (
    trust_long.groupby(["variable", "score"]).size().unstack(fill_value=0).reindex(columns=sorted(trust_long["score"].dropna().unique())).rename(columns=lambda c: f"count_{int(c)}")
)

summary_stats = (
    trust_long.groupby(["variable", "variable_label", "variable_label_with_id", "variable_label_wrapped"])
    .agg(mean_score=("score", "mean"), std_score=("score", "std"),
         min_score=("score", "min"), max_score=("score", "max"), n=("score", "count"))
    .reset_index()
    .assign(variable_order=lambda df: df["variable"].map({v: i for i, v in enumerate(TRUST_VARS)}))
    .sort_values("variable_order")
)
summary_stats = summary_stats.merge(missing_counts, left_on="variable", right_index=True)
summary_stats = summary_stats.merge(valid_counts, left_on="variable", right_index=True)
summary_stats["missing_rate"] = summary_stats["missing"] / (summary_stats["missing"] + summary_stats["n_valid"])
summary_stats = summary_stats.merge(score_counts, left_on="variable", right_index=True, how="left").fillna(0)

# Cross-country/year & demographic slices
country_year = trust_long.groupby(["country_name", "year_int", "variable_label_wrapped", "variable"]).agg(mean_score=("score", "mean")).reset_index()
year_overall = trust_long.groupby("year_int").agg(mean_score=("score", "mean")).reset_index().rename(columns={"year_int": "year"})

gender_slice = (
    trust_long.dropna(subset=["gender"])
    .loc[lambda df: ~df["gender"].str.strip().str.lower().str.contains(r"missing|decline|don.?t know|can.?t choose|do\s*not\s*under\s*stand", na=False)]
    .groupby(["variable", "variable_label_wrapped", "gender"])
    .agg(mean_score=("score", "mean"))
    .reset_index()
    .assign(variable_order=lambda df: df["variable"].map({v: i for i, v in enumerate(TRUST_VARS)}))
    .sort_values("variable_order")
)

age_slice = trust_long.dropna(subset=["age"]).query("age < 100")
age_slice = age_slice[age_slice["country_name"].isin(AGE_COUNTRIES)]
age_slice["variable_label_wrapped"] = pd.Categorical(age_slice["variable_label_wrapped"], categories=VAR_ORDER_WRAPPED, ordered=True)

education_slice = trust_long.dropna(subset=["education"])
education_slice = education_slice[~education_slice["education"].str.strip().str.lower().str.contains(r"can.?t choose|decline|do\s*not\s*under\s*stand", na=False)]
education_slice = education_slice[education_slice["country_name"].isin(AGE_COUNTRIES)]
education_slice["variable_label_wrapped"] = pd.Categorical(education_slice["variable_label_wrapped"], categories=VAR_ORDER_WRAPPED, ordered=True)

display(summary_stats.head())

,variable,variable_label,variable_label_with_id,variable_label_wrapped,mean_score,std_score,min_score,max_score,n,variable_order,missing,n_valid,missing_rate,count_1,count_1,count_2,count_2,count_3,count_3,count_4
0,ti1,Trust in the president/Prime Minister,ti1 – Trust in the president/Prime Minister,ti1 – Trust in the<br>president/Prime<br>Minister,2.729585,1.059535,1.0,4.0,11046,0,2773,11046,0.200666,1819,0,2640,0,3296,0,3291
4,ti2,The national government,ti2 – The national government,ti2 – The national<br>government,2.485899,1.011043,1.0,4.0,7588,1,6231,7588,0.450901,1574,0,2109,0,2549,0,1356
5,ti3,Parliament,ti3 – Parliament,ti3 – Parliament,2.449436,1.046344,1.0,4.0,12489,2,1330,12489,0.096244,2890,0,3497,0,3701,0,2401
6,ti4,Local government,ti4 – Local government,ti4 – Local<br>government,2.487968,0.992314,1.0,4.0,10763,3,3056,10763,0.221145,2122,0,3099,0,3710,0,1832
7,ti5,The courts,ti5 – The courts,ti5 – The courts,2.609171,1.026626,1.0,4.0,11427,4,2392,11427,0.173095,2048,0,2983,0,3783,0,2613


In [283]:
# Build Plotly figures


# Custom blue scale for heatmap (lighter blues)
MINT_SCALE = px.colors.sequential.Blues[1:6]


# Ensure ordered categories for facets


# Deduplicate in case upstream merges add overlapping count_ columns
summary_stats = summary_stats.loc[:, ~summary_stats.columns.duplicated()]


country_var = (
    trust_long.groupby(["variable", "variable_label_wrapped", "country_name"])
    .agg(mean_score=("score", "mean"), n=("score", "count"))
    .reset_index()
 )
country_var["variable_label_wrapped"] = pd.Categorical(country_var["variable_label_wrapped"], categories=VAR_ORDER_WRAPPED, ordered=True)


year_var = (
    trust_long.groupby(["year_int", "variable", "variable_label_wrapped"])
    .agg(mean_score=("score", "mean"))
    .reset_index()
 )
year_var["variable_label_wrapped"] = pd.Categorical(year_var["variable_label_wrapped"], categories=VAR_ORDER_WRAPPED, ordered=True)


# 1) Overall means (ti codes only)
fig_overview = px.bar(
    summary_stats.sort_values("mean_score"),
    x="mean_score", y="variable", orientation="h",
    error_x="std_score", title="Mean trust scores (ti1–ti12)",
    labels={"mean_score": "Mean (0-4)", "variable": "ti"},
    category_orders={"variable": TRUST_VARS},
    color_discrete_sequence=PALETTE,
 )
fig_overview.update_layout(height=760, bargap=0.35, yaxis=dict(automargin=True))


# 2) Country × variable heatmap (all countries, ti codes only)
heatmap_df = country_year.assign(variable_code=lambda df: df["variable"])
heatmap_df = heatmap_df.pivot_table(
    index="country_name", columns="variable_code", values="mean_score", aggfunc="mean"
 )
heatmap_df = heatmap_df.reindex(columns=TRUST_VARS)
fig_heatmap = px.imshow(
    heatmap_df,
    color_continuous_scale=MINT_SCALE,
    aspect="auto",
    labels=dict(color="Mean"),
    title="All countries: mean trust scores (ti1–ti12)",
 )
fig_heatmap.update_layout(height=720)


# 3) Annual overall trend (integer years)
fig_trend = px.line(
    year_overall.sort_values("year"),
    x="year", y="mean_score", markers=True,
    title="Global trust trend over time",
    labels={"mean_score": "Mean trust", "year": "Year"},
    color_discrete_sequence=PALETTE,
 )
fig_trend.update_xaxes(dtick=1)


# 4) Gender slice (all institutions, pastel purple/green)
gender_colors = {
    "Female": "#d7c4f6",
    "Male": "#b5e8c3",
    "Other": "#c7e8ff",
}
fig_gender = px.bar(
    gender_slice,
    x="variable_label_wrapped", y="mean_score", color="gender", barmode="group",
    category_orders={"variable_label_wrapped": VAR_ORDER_WRAPPED},
    title="Institutional trust × gender (ti1–ti12)",
    labels={"variable_label_wrapped": "Institution", "mean_score": "Mean", "gender": "Gender"},
    color_discrete_map=gender_colors,
 )
fig_gender.update_layout(xaxis_tickangle=20, height=760)


# 5) Age vs. trust (heatmap country subset retained for readability)
age_countries_plot = list(AGE_COUNTRIES)
age_slice_plot = age_slice[age_slice["country_name"].isin(age_countries_plot)]
fig_age = px.scatter(
    age_slice_plot,
    x="age", y="score", color="country_name", facet_col="variable_label_wrapped", facet_col_wrap=3, facet_row_spacing=0.12,
    opacity=0.0,
    trendline="lowess",
    category_orders={"variable_label_wrapped": VAR_ORDER_WRAPPED},
    title="Age × institutional trust (heatmap country set)",
    labels={"score": "Trust score", "age": "Age", "country_name": "Country", "variable_label_wrapped": "Institution"},
    color_discrete_sequence=PALETTE,
 )
fig_age.update_traces(marker=dict(opacity=0))
fig_age.update_xaxes(dtick=10, tickformat=".0f")
fig_age.update_layout(height=1320)


# 6) Education × trust (heatmap country set, ordered levels)
education_countries_plot = list(AGE_COUNTRIES)
education_slice_plot = education_slice[education_slice["country_name"].isin(education_countries_plot)]
education_levels = [educ_map[k] for k in sorted(educ_map.keys(), key=lambda x: str(x)) if educ_map.get(k) in education_slice_plot["education"].unique()]
fig_education = px.bar(
    education_slice_plot,
    x="education", y="score", color="country_name",
    facet_col="variable_label_wrapped", facet_col_wrap=3, facet_row_spacing=0.12,
    barmode="group",
    category_orders={"education": education_levels, "variable_label_wrapped": VAR_ORDER_WRAPPED},
    title="Education × institutional trust (heatmap country set)",
    labels={"score": "Trust score", "education": "Education", "country_name": "Country", "variable_label_wrapped": "Institution"},
    color_discrete_sequence=px.colors.qualitative.Set1,
 )
fig_education.update_layout(height=1320)
fig_education.update_xaxes(tickfont={"size": 10}, automargin=True, tickmode="array", tickvals=education_levels, ticktext=education_levels)


# 7) Distribution of trust scores by institution (all ti1–ti12; ti labels only)
fig_distribution = px.box(
    trust_long,
    x="variable", y="score", color="variable", points="outliers", notched=True,
    category_orders={"variable": TRUST_VARS},
    title="Distribution of trust scores by institution",
    labels={"variable": "ti", "score": "Trust score"},
    color_discrete_sequence=PALETTE,
 )
fig_distribution.update_layout(showlegend=False, height=900, xaxis_tickangle=0)


# 8) Variation across countries (means by institution and country; selected facets remain)
fig_country_var = px.bar(
    country_var,
    x="mean_score", y="country_name", color="variable_label_wrapped", orientation="h",
    facet_col="variable_label_wrapped", facet_col_wrap=3, facet_row_spacing=0.08,
    category_orders={"variable_label_wrapped": VAR_ORDER_WRAPPED},
    title="Institutional trust across countries (mean scores)",
    labels={"mean_score": "Mean trust", "country_name": "Country", "variable_label_wrapped": "Institution"},
    color_discrete_sequence=PALETTE,
 )
fig_country_var.update_layout(height=1200, legend_title="Institution")


# 9) Variation across all countries (means per ti across all countries)
country_means_all = (
    trust_long.groupby(["variable", "country_name"]).agg(mean_score=("score", "mean")).reset_index()
 )
fig_country_all = px.box(
    country_means_all,
    x="variable", y="mean_score", color="variable",
    category_orders={"variable": TRUST_VARS},
    title="Cross-country variation of institutional trust (ti1–ti12)",
    labels={"variable": "ti", "mean_score": "Mean trust"},
    color_discrete_sequence=PALETTE,
 )
fig_country_all.update_layout(showlegend=False, height=780, xaxis_tickangle=0)


# 10) Variation across years (means by institution and year; ti order)
fig_year_var = px.line(
    year_var.sort_values(["year_int", "variable_label_wrapped"]),
    x="year_int", y="mean_score", color="variable_label_wrapped", markers=True,
    facet_col="variable_label_wrapped", facet_col_wrap=3, facet_row_spacing=0.08,
    category_orders={"variable_label_wrapped": VAR_ORDER_WRAPPED},
    title="Institutional trust across years (mean scores)",
    labels={"year_int": "Year", "mean_score": "Mean trust", "variable_label_wrapped": "Institution"},
    color_discrete_sequence=PALETTE,
 )
fig_year_var.update_xaxes(dtick=1)
fig_year_var.update_layout(height=1200, legend_title="Institution")


# 11) Demographic associations: education (all institutions, pastel to match income)
education_demo = (
    trust_long.dropna(subset=["education"])
.loc[lambda df: ~df["education"].str.strip().str.lower().str.contains(r"can.?t choose|decline|do\s*not\s*under\s*stand", na=False)]
 )
education_demo = (
    education_demo.groupby(["variable_label_wrapped", "education"])
    .agg(mean_score=("score", "mean"), n=("score", "count"))
    .reset_index()
 )
education_order = [educ_map[k] for k in sorted(educ_map.keys(), key=lambda x: str(x)) if educ_map.get(k) in education_demo["education"].unique()]
fig_demo_education = px.bar(
    education_demo,
    x="education", y="mean_score", color="variable_label_wrapped", barmode="group",
    category_orders={"education": education_order, "variable_label_wrapped": VAR_ORDER_WRAPPED},
    title="Political trust by education (ti1–ti12)",
    labels={"education": "Education", "mean_score": "Mean trust", "variable_label_wrapped": "Institution"},
    color_discrete_sequence=px.colors.qualitative.Pastel1,
 )
fig_demo_education.update_layout(height=720, legend_title="Institution")
fig_demo_education.update_xaxes(tickmode="array", tickvals=education_order, ticktext=education_order)


# 12) Demographic associations: income (all institutions, pastel)
income_demo = (
    trust_long.dropna(subset=["income"])
    .loc[lambda df: ~df["income"].str.strip().str.lower().str.contains(r"missing|decline|don.?t know|can.?t choose|do\s*not\s*under\s*stand", na=False)]
 )
income_demo = (
    income_demo.groupby(["variable_label_wrapped", "income"])
    .agg(mean_score=("score", "mean"), n=("score", "count"))
    .reset_index()
 )
income_order = sorted(income_demo["income"].dropna().unique(), key=lambda x: str(x))
fig_demo_income = px.bar(
    income_demo,
    x="income", y="mean_score", color="variable_label_wrapped", barmode="group",
    category_orders={"income": income_order, "variable_label_wrapped": VAR_ORDER_WRAPPED},
    title="Political trust by income (ti1–ti12)",
    labels={"income": "Income", "mean_score": "Mean trust", "variable_label_wrapped": "Institution"},
    color_discrete_sequence=px.colors.qualitative.Pastel1,
 )
fig_demo_income.update_layout(height=720, legend_title="Institution", xaxis_tickangle=20)

In [284]:
# Render HTML dashboard
OUTPUT_HTML.parent.mkdir(parents=True, exist_ok=True)




def fig_fragment(fig: go.Figure) -> str:
    return fig.to_html(full_html=False, include_plotlyjs=False, config={"displaylogo": False})




overview_html = fig_fragment(fig_overview)
distribution_html = fig_fragment(fig_distribution)
country_all_html = fig_fragment(fig_country_all)
heatmap_html = fig_fragment(fig_heatmap)
trend_html = fig_fragment(fig_trend)
gender_html = fig_fragment(fig_gender)
age_html = fig_fragment(fig_age)
education_html = fig_fragment(fig_education)
country_var_html = fig_fragment(fig_country_var)
year_var_html = fig_fragment(fig_year_var)
demo_education_html = fig_fragment(fig_demo_education)
demo_income_html = fig_fragment(fig_demo_income)


summary_display = summary_stats.copy()
summary_display[["mean_score", "std_score", "min_score", "max_score", "missing_rate"]] = summary_display[["mean_score", "std_score", "min_score", "max_score", "missing_rate"]].round(3)
score_cols = [c for c in summary_display.columns if str(c).startswith("count_")]
summary_display = summary_display.assign(variable_order=lambda df: df["variable"].map({v: i for i, v in enumerate(TRUST_VARS)}))
summary_display = summary_display.sort_values("variable_order")
summary_display = summary_display[["variable", "variable_label", "mean_score", "std_score", "min_score", "max_score", "n", "missing", "missing_rate"] + score_cols]
summary_html = summary_display.to_html(index=False, classes="table table-striped", justify="center")


heat_note = "All countries in this sample are shown; ti columns follow ti1–ti12 so institutional contrasts stay aligned."


analysis_overview = (
    "Analysis: Overall means compress twelve institutions into a horizontal ranking, highlighting which nodes of the state attract relatively greater confidence. "
    "In this sample, comparing means is diagnostic rather than causal: higher values may reflect performance, propaganda, or selective answering. "
    "The error bars signal dispersion, warning that similar means can mask heterogeneous sub-publics. "
    "Because missing codes are removed, averages lean toward respondents willing to express trust; this should temper over-generalization."
 )


analysis_distribution = (
    "Analysis: Boxplots show the full distribution of trust responses for each institution, surfacing asymmetry, floor or ceiling effects, and outliers. "
    "Tight boxes with high medians suggest concentrated confidence; wide boxes with low medians highlight fragmentation or skepticism."
 )


analysis_heat = (
    "Analysis: The heatmap spans every available country, aligning ti1–ti12 so institutional contrasts dominate. "
    "High uniformity across columns may indicate generalized regime support; sharp vertical contrasts suggest institution-specific legitimacy gaps. "
    "Missingness by country-item can depress means; hover to see exact values."
 )


analysis_country_all = (
    "Analysis: Cross-country variation boxes summarize how national means distribute for each ti code, showing whether dispersion is driven by country differences or institutional baselines. "
    "Narrow boxes imply consensus across countries; wide boxes signal country-specific trust gaps."
 )


analysis_country_var = (
    "Analysis: Country facets map institutional trust to national means, revealing whether variation is driven more by countries or institutions. "
    "Comparing bars within a facet indicates cross-country spread for a given institution; cross-facet comparison shows institution-level baselines."
 )


analysis_trend = (
    "Analysis: The global mean line aggregates across countries and institutions, so level shifts often trace compositional change rather than synchronized sentiment. "
    "Integer years remove spurious mid-year noise, focusing attention on sustained inflections. "
    "Reading the slope critically: modest drifts can still be policy-relevant when persistent; sharp breaks often imply exogenous shocks or sampling pivots."
 )


analysis_year_var = (
    "Analysis: Year facets show mean trajectories by institution, indicating whether temporal drift is broad-based or institution-specific. "
    "Parallel slopes hint at common shocks; diverging slopes imply differentiated legitimacy dynamics."
 )


analysis_gender = (
    "Analysis: Gender gaps across all twelve institutions reveal how safety, representation, and fairness perceptions diverge. "
    "Grouped bars make asymmetries visible; persistent male-female splits on police or parties can signal differentiated exposure to coercion or clientelism. "
    "Interpretation should consider intersectionality (age, class) that is not shown here, marking a frontier for deeper models."
 )


analysis_age = (
    "Analysis: LOWESS lines trace how trust co-moves with age without assuming linearity. "
    "Removing markers de-emphasizes noise and highlights structural age gradients that could reflect cohort socialization or life-cycle effects. "
    "Country colors separate contexts; parallel slopes hint at common mechanisms, while crossings flag context-specific trajectories. "
    "Facets use the heatmap subset to keep visual comparability tight; x-axis ticks show ages explicitly."
 )


analysis_education = (
    "Analysis: Education facets apply the same country subset, grouping trust by schooling levels across all institutions. "
    "Differences across education can signal information access, civic skills, or class-linked experiences with the state. "
    "Grouped bars facilitate cross-country contrasts within each institutional dimension while keeping barometer coverage consistent."
 )


analysis_demo = (
    "Analysis: Demographic slices here focus on education and income, keeping categories consistent with the socio-economic sheet. "
    "Differences across schooling or income can signal resource access, exposure, or status-linked experiences with the state."
 )


analysis_desc = (
    "Analysis: The descriptive table orders ti1–ti12 to preserve the institutional sequence, pairing moments with dispersion and missingness. "
    "High missing rates may signal question sensitivity or survey fatigue, cautioning against naive comparisons. "
    "Min–max bounds uncover ceiling or floor effects that can compress variance and distort regressions."
 )


gender_note = "All ti1–ti12 included; grouped bars highlight gender gaps per institution."


age_note = (
    f"Age facets use the heatmap country subset (actual plotted: {', '.join(list(AGE_COUNTRIES))}) for legibility, "
    "mirroring the barometer coverage rationale: Mainland China (ABS3), India (SAB), Brazil (LATINO), Nigeria (AFRO5), "
    "Russia (EURASIA), Egypt (ARAB3), South Korea (ABS3), South Africa (AFRO5)."
    "<ul style='margin:6px 0 0 0; padding-left:18px;'>"
    "<li>Mainland China – ABS3; authoritarian benchmark with persistently high trust.</li>"
    "<li>India – SAB; largest democracy, diverse society, South Asia reference.</li>"
    "<li>Brazil – LATINO; post-transition democracy with corruption shocks.</li>"
    "<li>Nigeria – AFRO5; populous, multi-ethnic state, SSA governance challenges.</li>"
    "<li>Russia – EURASIA; hybrid regime with recent volatility.</li>"
    "<li>Egypt – ARAB3; Arab Spring and subsequent shifts.</li>"
    "<li>South Korea – ABS3; mature democracy with high development.</li>"
    "<li>South Africa – AFRO5; post-transition trust swings for comparison.</li>"
    "</ul>"
 )


education_note = (
    f"Education facets mirror the heatmap country subset (actual plotted: {', '.join(list(AGE_COUNTRIES))}); grouped bars by schooling support cross-country comparison within each institution."
 )


demo_note = (
    "Demographic charts here focus on education and income; rare missing or decline options are removed for clarity."
 )


html_page = f"""
<!DOCTYPE html>
<html lang='en'>
<head>
  <meta charset='UTF-8'>
  <title>Global Political Trust Dashboard</title>
  <script src='https://cdn.plot.ly/plotly-2.32.0.min.js'></script>
  <style>
    body {{ font-family: \"Times New Roman\", \"Times\", serif; margin: 0; background: #f7f8fb; color: #1f2933; }}
    h1, h2 {{ margin: 0 0 12px 0; }}
    .hero {{ padding: 32px; background: linear-gradient(120deg, #fbe7e2, #e7f3ff, #eaf7f0); color: #1f2933; }}
    .grid {{ display: grid; gap: 18px; padding: 18px; }}
    .grid.one {{ grid-template-columns: 1fr; }}
    .card {{ background: #ffffff; border: 1px solid #e5e7eb; border-radius: 14px; padding: 16px; box-shadow: 0 12px 28px rgba(31,41,51,0.07); }}
    .pill {{ display: inline-flex; align-items: center; gap: 6px; padding: 6px 10px; background: #f1f5f9; border-radius: 999px; font-size: 12px; color: #475569; }}
    table {{ width: 100%; border-collapse: collapse; color: #1f2933; }}
    th, td {{ border: 1px solid #e5e7eb; padding: 8px 10px; }}
    th {{ background: #f1f5f9; }}
    tr:nth-child(even) {{ background: #f9fafb; }}
    a {{ color: #3b82f6; }}
    .section-title {{ display:flex; justify-content:space-between; align-items:center; }}
    .note {{ margin: 12px 0 0 0; color: #475569; font-size: 13px; line-height: 1.5; }}
    .analysis {{ margin: 10px 0 0 0; color: #2b2b2b; font-size: 14px; line-height: 1.55; }}
  </style>
</head>
<body>
  <div class='hero'>
    <p class='pill'>GBS 2.9 · Political Trust</p>
    <h1>Global Political Trust Dashboard</h1>
    <p>Structured flow: descriptive stats first, then distributions, all-country heatmap, country/year patterns, demographic associations, and focused visuals for the selected country subset.</p>
  </div>
  <div class='grid one'>
    <div class='card'>
      <div class='section-title'><h2>Descriptive stats</h2><span class='pill'>ti1–ti12</span></div>
      {summary_html}
      <p class='note'>Means exclude invalid codes (-1/0/7/8/9). Columns count_1–count_4 show valid response counts for each score.</p>
      <p class='analysis'>{analysis_desc}</p>
    </div>
    <div class='card'>
      <div class='section-title'><h2>Overall means (ti1–ti12)</h2><span class='pill'>Mean + SD</span></div>
      {overview_html}
      <p class='note'>ti codes only; institutional trust items ti1–ti12. Scale 1–4 as noted above.</p>
      <p class='analysis'>{analysis_overview}</p>
    </div>
    <div class='card'>
      <div class='section-title'><h2>Distribution of responses</h2><span class='pill'>Boxplots</span></div>
      {distribution_html}
      <p class='note'>Interactive boxplots for each ti code; order ti1–ti12.</p>
      <p class='analysis'>{analysis_distribution}</p>
    </div>
    <div class='card'>
      <div class='section-title'><h2>Country heatmap</h2><span class='pill'>All countries</span></div>
      {heatmap_html}
      <p class='note'>{heat_note}</p>
      <p class='analysis'>{analysis_heat}</p>
    </div>
    <div class='card'>
      <div class='section-title'><h2>Cross-country variation (all countries)</h2><span class='pill'>ti1–ti12</span></div>
      {country_all_html}
      <p class='note'>Boxes show dispersion of country means for each ti code.</p>
      <p class='analysis'>{analysis_country_all}</p>
    </div>
    <div class='card'>
      <div class='section-title'><h2>Variation across countries</h2><span class='pill'>Means by country</span></div>
      {country_var_html}
      <p class='note'>Mean scores by institution and country; facet panels align ti order.</p>
      <p class='analysis'>{analysis_country_var}</p>
    </div>
    <div class='card'>
      <div class='section-title'><h2>Yearly trend</h2><span class='pill'>Global mean</span></div>
      {trend_html}
      <p class='note'>Years coerced to integers to remove mid-year artifacts.</p>
      <p class='analysis'>{analysis_trend}</p>
    </div>
    <div class='card'>
      <div class='section-title'><h2>Variation across years</h2><span class='pill'>Means by year</span></div>
      {year_var_html}
      <p class='note'>Mean scores by institution over time; facets ordered ti1–ti12.</p>
      <p class='analysis'>{analysis_year_var}</p>
    </div>
    <div class='card'>
      <div class='section-title'><h2>Gender slice</h2><span class='pill'>All institutions</span></div>
      {gender_html}
      <p class='note'>{gender_note}</p>
      <p class='analysis'>{analysis_gender}</p>
    </div>
    <div class='card'>
      <div class='section-title'><h2>Age × trust</h2><span class='pill'>Heatmap country subset</span></div>
      {age_html}
      <p class='note'>{age_note}</p>
      <p class='analysis'>{analysis_age}</p>
    </div>
    <div class='card'>
      <div class='section-title'><h2>Education × trust</h2><span class='pill'>Heatmap country subset</span></div>
      {education_html}
      <p class='note'>{education_note}</p>
      <p class='analysis'>{analysis_education}</p>
    </div>
    <div class='card'>
      <div class='section-title'><h2>Demographics × trust</h2><span class='pill'>Education, income</span></div>
      <div style='display:grid; gap:12px;'>
        {demo_education_html}
        {demo_income_html}
      </div>
      <p class='note'>{demo_note}</p>
      <p class='analysis'>{analysis_demo}</p>
    </div>
  </div>
</body>
</html>
"""


OUTPUT_HTML.write_text(html_page, encoding="utf-8")
print(f"Dashboard saved to → {OUTPUT_HTML.resolve()}")

Dashboard saved to → /Users/lisic/Library/CloudStorage/GoogleDrive-u3629075@connect.hku.hk/我的云端硬盘/Lab Survey AI Analysis/report/gbs_pt_html.html
